### Импорты и загрузка

In [1]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
from pathlib import Path

from dataset import build_item_vocab, build_user_sequences, NextItemDataset

PROC = Path('../data/processed')
train = pd.read_parquet(PROC / 'train.parquet')
print(f"Train: {len(train):,} rows, {train['user_id'].nunique():,} users")

Train: 3,787,046 rows, 727,645 users


### Словарь

In [2]:
item2idx, idx2item = build_item_vocab(train)

print(f"Уникальных айтемов: {len(item2idx):,}")
print(f"Vocab size (с PAD и UNK): {len(item2idx) + 2:,}")

# Проверка нескольких маппингов
sample_items = list(item2idx.items())[:5]
print(f"\nПервые 5 маппингов (item_id -> index):")
for item, idx in sample_items:
    print(f"  {item} -> {idx}")

# И обратный
for i in range(2, 7):
    print(f"  index {i} -> item_id {idx2item[i]}")

# Проверка консистентности
for item, idx in sample_items:
    assert idx2item[idx] == item, f"Mismatch: {item} -> {idx} -> {idx2item[idx]}"
print("\n Маппинг согласован")

Уникальных айтемов: 14,935
Vocab size (с PAD и UNK): 14,937

Первые 5 маппингов (item_id -> index):
  0 -> 2
  1 -> 3
  2 -> 4
  3 -> 5
  4 -> 6
  index 2 -> item_id 0
  index 3 -> item_id 1
  index 4 -> item_id 2
  index 5 -> item_id 3
  index 6 -> item_id 4

 Маппинг согласован


### Последовательности

In [3]:
sequences = build_user_sequences(train, max_len=50)
print(f"Пользователей с историей: {len(sequences):,}")

seq_lengths = pd.Series([len(s) for s in sequences.values()])
print(f"\nРаспределение длин последовательностей:")
print(seq_lengths.describe(percentiles=[.25, .5, .75, .9, .99]))

print(f"\nЮзеров с 1 айтемом: {(seq_lengths == 1).sum():,} ({(seq_lengths == 1).mean()*100:.1f}%)")
print(f"Юзеров с 2+ айтемами: {(seq_lengths >= 2).sum():,}")
print(f"Юзеров с 50 айтемами (обрезка): {(seq_lengths == 50).sum():,}")

Пользователей с историей: 727,645

Распределение длин последовательностей:
count    727645.000000
mean          4.983838
std           7.424481
min           1.000000
25%           1.000000
50%           2.000000
75%           5.000000
90%          12.000000
99%          44.000000
max          50.000000
dtype: float64

Юзеров с 1 айтемом: 276,525 (38.0%)
Юзеров с 2+ айтемами: 451,120
Юзеров с 50 айтемами (обрезка): 5,510


### Dataset и один пример

In [4]:
ds = NextItemDataset(sequences, item2idx, max_len=50)
print(f"Обучающих примеров: {len(ds):,}")

# Один пример
sample = ds[0]
print(f"\nФорма input_ids: {sample['input_ids'].shape}")
print(f"Форма labels:    {sample['labels'].shape}")

# Первые 15 позиций
inp = sample['input_ids'][:15].tolist()
lbl = sample['labels'][:15].tolist()
print(f"\nПервые 15 позиций:")
print(f"  input_ids: {inp}")
print(f"  labels:    {lbl}")

# Проверка логики
n_pad = (sample['input_ids'] == 0).sum().item()
print(f"\nPAD-позиций: {n_pad}")

# Проверка: labels[i] должен равняться input_ids[i] везде, кроме PAD-позиций
input_ids_np = sample['input_ids'].numpy()
labels_np = sample['labels'].numpy()
non_pad_mask = input_ids_np != 0
assert np.array_equal(input_ids_np[non_pad_mask], labels_np[non_pad_mask]), "labels != input_ids вне PAD"
print("labels совпадают с input_ids везде, кроме PAD-позиций")

# Проверка: labels в PAD-позициях должен быть -100
pad_mask = input_ids_np == 0
assert (labels_np[pad_mask] == -100).all(), "PAD-позиции в labels должны быть -100"
print("В PAD-позициях labels = -100")

Обучающих примеров: 451,120

Форма input_ids: torch.Size([50])
Форма labels:    torch.Size([50])

Первые 15 позиций:
  input_ids: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  labels:    [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]

PAD-позиций: 44
labels совпадают с input_ids везде, кроме PAD-позиций
В PAD-позициях labels = -100


*PAD-ы заменились в labels с 0 до -100, как и ожидалось. Ошибок нет, всё хорошо.*
*44 PAD-позиции из 50 - норма, пользователи в среднем имеют в истории 5 (4,98) таргетов*